# Introduction:

*Stuff to add*

- Contents structure
- Explain what the project is about
- How we carried out the analysis
- What are the tools used



## Data Acquisition:

### Primary: Guardian API
We will collect articles from the Guardian API using commodity-related search terms (e.g. oil, natural gas, copper, wheat, OPEC, energy crisis) filtered by relevant sections (e.g. Business, Environment, World News). For each article, we may extract the publication date, headline, section, tags, word count, and body text. We plan to collect data spanning approximately 2014 - 2026 to capture multiple geopolitical cycles.

### Secondary: Commodity Price Data
To contextualise media coverage against market movements, we will source historical commodity price data from Yahoo Finance (via the yfinance Python library) or publicly available datasets. This will include daily/weekly prices for key commodities such as Brent Crude, natural gas, gold, copper, and wheat.

- Query the Guardian API with commodity-related keywords and display results across the full date range

- Download commodity price time series from Yahoo Finance using yfinance

- Store raw data in structured formats (JSON/CSV) for reproducibility



In [ ]:
import json
import requests
import pandas as pd
import time



with open('keys.json') as f:
    key = json.load(f)

API_KEY = key['guardian']['api_key']
BASE_URL = 'https://content.guardianapis.com'

# generate (first_day, last_day) pairs for each month
month_starts = pd.date_range("2020-01-01", "2026-03-31", freq="MS")
date_ranges = [
    (d.strftime("%Y-%m-%d"), (d + pd.offsets.MonthEnd(0)).strftime("%Y-%m-%d"))
    for d in month_starts
]

ARTICLES_PER_MONTH = 50
all_results = []



for from_date, to_date in date_ranges:
    for page in range(1, (ARTICLES_PER_MONTH // 50) + 1):
        parameters = {
            "api-key": API_KEY,
            "q": "oil OR natural gas OR gold OR OPEC OR energy crisis",
            "page-size": ARTICLES_PER_MONTH,
            "page": page,
            "show-fields": "bodyText",
            "from-date": from_date,
            "to-date": to_date,
            "order-by": "relevance"
        }
        response = requests.get(f"{BASE_URL}/search", params=parameters)

        data = response.json()['response']

        for article in data['results']:
            all_results.append({
                'date': article.get('webPublicationDate'),
                'section': article.get('sectionName'),
                'title': article.get('webTitle'),
            })

        if page >= data['pages']:
            break
        time.sleep(0.5)


print(f"Done.  Total: {len(all_results)}")



In [ ]:
import yfinance as yf


# Define commodity tickers
# Yahoo Finance uses these symbols for common commodities:
# all futures
commodities = {
    "Gold":        "GC=F",   
    "Brent Oil":   "BZ=F",   # brent crude oil
    "Natural Gas": "NG=F",   
    "Wheat":       "ZW=F",   
}

# Download historical data
data = yf.download(
    tickers=list(commodities.values()),
    start="2020-01-01",
    end="2025-12-31",
    interval="1wk"       # 1d for daily, 1wk for weekly, 1mo for monthly
)

# Extract closing prices only
close_prices = data["Close"]
close_prices.columns = list(commodities.keys())

df_commodities_prices = pd.DataFrame(close_prices)


print(close_prices.head(50))


# IDA

In [ ]:
# mantra 

# Data wrangling

# EDA  

# Visualisation

rian